# Stage 3 — Real Persona Dataset Analysis

**Project:** Persona Engine — Stage 3
**Objective:** Visualize real WhatsApp conversational distributions, persona response styles, context depths, and linguistic signals without running model training.
**Data Source:** `ml/data/processed/raw_audit.json`, `ml/data/processed/persona_statistics.json`, `ml/data/training/train.jsonl`

In [ ]:
import sys
import json
from pathlib import Path
from collections import Counter

# Set up workspace path
ml_root = Path.cwd().resolve()
if ml_root.name != "ml" and (ml_root / "ml").is_dir():
    ml_root = ml_root / "ml"
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

print("ML Root:", ml_root)

## 1. Load Processed Dataset Statistics

In [ ]:
audit_path = ml_root / "data" / "processed" / "raw_audit.json"
stats_path = ml_root / "data" / "processed" / "persona_statistics.json"

with open(audit_path, "r", encoding="utf-8") as f:
    raw_audit = json.load(f)

with open(stats_path, "r", encoding="utf-8") as f:
    persona_stats = json.load(f)

print("Loaded raw audit and persona statistics.")
print("Total Parsed Messages:", f"{raw_audit['total_parsed_messages']:,}")
print("Total Candidates:", f"{persona_stats['candidate_dataset_metrics']['total_candidates']:,}")

## 2. Speaker Balance & Activity

In [ ]:
speakers = raw_audit["messages_per_speaker"]
print("=" * 40)
print("SPEAKER MESSAGE DISTRIBUTION")
print("=" * 40)
for spk, cnt in speakers.items():
    bar = "#" * int(cnt / 150)
    print(f"{spk:15} | {cnt:5,} | {bar}")

## 3. Message Length & Persona Response Lengths

In [ ]:
len_dist = persona_stats["persona_speaker_overview"]["length_distribution"]
print("=" * 40)
print("PERSONA RESPONSE LENGTH CATEGORIES")
print("=" * 40)
for cat, cnt in len_dist.items():
    bar = "#" * int(cnt / 50)
    print(f"{cat:25} | {cnt:5,} | {bar}")

avg_len = persona_stats["persona_speaker_overview"]["average_response_length_chars"]
med_len = persona_stats["persona_speaker_overview"]["median_response_length_chars"]
print(f"\nAverage Length: {avg_len} chars | Median Length: {med_len} chars")

## 4. Context Window Depth Distribution

In [ ]:
ctx_dist = persona_stats["candidate_dataset_metrics"]["context_window_distribution"]
print("=" * 40)
print("CONTEXT WINDOW DISTRIBUTIONS")
print("=" * 40)
for turns, cnt in sorted(ctx_dist.items(), key=lambda x: int(x[0])):
    bar = "#" * int(cnt / 80)
    print(f"{turns}-turn context: {cnt:5,} examples | {bar}")

## 5. Linguistic & Response Type Breakdown

In [ ]:
lang_dist = persona_stats["candidate_dataset_metrics"]["language_distribution"]
resp_dist = persona_stats["candidate_dataset_metrics"]["response_type_distribution"]

print("=" * 40)
print("LANGUAGE DISTRIBUTION")
print("=" * 40)
for lang, cnt in lang_dist.items():
    print(f"{lang:15} | {cnt:5,} examples")

print("\n" + "=" * 40)
print("RESPONSE TYPE DISTRIBUTION")
print("=" * 40)
for rt, cnt in resp_dist.items():
    print(f"{rt:15} | {cnt:5,} examples")

## 6. Top Emojis & Top Words

In [ ]:
print("=" * 40)
print("TOP 10 EMOJIS USED BY PERSONA")
print("=" * 40)
for em, cnt in persona_stats["persona_speaker_overview"]["top_emojis"][:10]:
    print(f"{em} : {cnt} times")

print("\n" + "=" * 40)
print("TOP 15 WORDS / LEXICAL MARKERS")
print("=" * 40)
for w, cnt in persona_stats["persona_speaker_overview"]["top_words"][:15]:
    print(f"{w:12} : {cnt:4,} occurrences")

## 7. Zero Data Leakage Split Verification

In [ ]:
splits = persona_stats["splits"]
print("=" * 40)
print("TRAIN / VAL / TEST SPLIT SIZES")
print("=" * 40)
for s_name in ["train", "validation", "test"]:
    info = splits[s_name]
    print(f"{s_name.capitalize():12} | {info['conversations']:4} convs | {info['examples']:5,} examples ({info['percentage_examples']} %)")

print(f"\nZero Data Leakage Status: {'PASSED (0% Overlap)' if splits['data_leakage_free'] else 'FAILED'}")